In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [16]:
df = pd.read_csv(r'Titanic-Dataset.csv')
print(df.shape) 


(891, 12)


In [17]:
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
230,231,1,1,"Harris, Mrs. Henry Birkhardt (Irene Wallach)",female,35.0,1,0,36973,83.475,C83,S
630,631,1,1,"Barkworth, Mr. Algernon Henry Wilson",male,80.0,0,0,27042,30.000,A23,S
211,212,1,2,"Cameron, Miss. Clear Annie",female,35.0,0,0,F.C.C. 13528,21.000,NaN,S
45,46,0,3,"Rogers, Mr. William John",male,NaN,0,0,S.C./A.4. 23567,8.050,NaN,S
472,473,1,2,"West, Mrs. Edwy Arthur (Ada Mary Worth)",female,33.0,1,2,C.A. 34651,27.750,NaN,S


In [18]:
df = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

In [19]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [20]:
df.sample(2)
df['Family_size'] = df['SibSp'] + df['Parch']
df = df.drop(['SibSp', 'Parch'], axis=1)

In [21]:
X = df.drop('Survived', axis=1)
y = df['Survived']

In [28]:

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [29]:
for i in range(len(X['Embarked'])):
    if X['Embarked'][i] == 'NAN':
        X['Embarked'][i] = 'S'


In [30]:
trf = ColumnTransformer([
    ('trf1', OrdinalEncoder(), ['Sex']),
    ('trf2', OneHotEncoder( drop='first'), ['Embarked']),
    ('trf3', SimpleImputer(strategy='mean'), ['Age'])
], remainder='passthrough')

In [31]:
pipe = Pipeline(steps =[
    ('step1', trf),
    ('Model',RandomForestClassifier(max_depth=9, n_estimators=50, random_state=42)),
])

In [32]:
pipe.fit(x_train, y_train)
y_pred = pipe.predict(x_test)

In [33]:
accuracy_score(y_test, y_pred)

0.8379888268156425

## Grid search CV

In [14]:
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [5, 10, None],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2]
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)

grid.fit(x_train, y_train)


ValueError: Invalid parameter 'model' for estimator Pipeline(steps=[('step1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('trf1', OrdinalEncoder(),
                                                  ['Sex']),
                                                 ('trf2',
                                                  OneHotEncoder(drop='first'),
                                                  ['Embarked']),
                                                 ('trf3', SimpleImputer(),
                                                  ['Age'])])),
                ('Model',
                 RandomForestClassifier(max_depth=9, n_estimators=50,
                                        random_state=42))]). Valid parameters are: ['memory', 'steps', 'transform_input', 'verbose'].